In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
from tqdm import tqdm


In [ ]:
# Danh sách các keypoint cần trích xuất theo yêu cầu của bạn
BODY_IDENTIFIERS = [
    "nose", "neck", "rightEye", "leftEye", "rightEar", "leftEar", 
    "rightShoulder", "leftShoulder", "rightElbow", "leftElbow", 
    "rightWrist", "leftWrist"
]

HAND_IDENTIFIERS = [
    "wrist", "indexTip", "indexDIP", "indexPIP", "indexMCP", 
    "middleTip", "middleDIP", "middlePIP", "middleMCP", 
    "ringTip", "ringDIP", "ringPIP", "ringMCP", 
    "littleTip", "littleDIP", "littlePIP", "littleMCP",
    "thumbTip", "thumbIP", "thumbMP", "thumbCMC"
]

TARGET_FRAMES = 204

In [ ]:
# Khởi tạo các giải pháp của MediaPipe
mp_holistic = mp.solutions.holistic

# --- MAPPING TÊN KEYPOINT SANG CHỈ SỐ CỦA MEDIAPIPE ---
POSE_MAPPING = {
    "nose": mp_holistic.PoseLandmark.NOSE.value,
    "rightEye": mp_holistic.PoseLandmark.RIGHT_EYE.value,
    "leftEye": mp_holistic.PoseLandmark.LEFT_EYE.value,
    "rightEar": mp_holistic.PoseLandmark.RIGHT_EAR.value,
    "leftEar": mp_holistic.PoseLandmark.LEFT_EAR.value,
    "rightShoulder": mp_holistic.PoseLandmark.RIGHT_SHOULDER.value,
    "leftShoulder": mp_holistic.PoseLandmark.LEFT_SHOULDER.value,
    "rightElbow": mp_holistic.PoseLandmark.RIGHT_ELBOW.value,
    "leftElbow": mp_holistic.PoseLandmark.LEFT_ELBOW.value,
    "rightWrist": mp_holistic.PoseLandmark.RIGHT_WRIST.value,
    "leftWrist": mp_holistic.PoseLandmark.LEFT_WRIST.value,
}

HAND_MAPPING = {name: i for i, name in enumerate(HAND_IDENTIFIERS)}


In [ ]:
def is_zero_frame(frame_data):
    """Kiểm tra xem một khung hình có phải toàn số 0 hay không."""
    return all(value == 0 for value in frame_data.values())

In [ ]:

def extract_specific_keypoints(results):
    """
    Trích xuất các keypoint cụ thể từ kết quả của MediaPipe và trả về một dictionary.
    """
    frame_keypoints = {}
    
    # --- Trích xuất keypoint CƠ THỂ ---
    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        for name, idx in POSE_MAPPING.items():
            frame_keypoints[f"{name}_X"] = landmarks[idx].x
            frame_keypoints[f"{name}_Y"] = landmarks[idx].y
        left_shoulder = landmarks[mp_holistic.PoseLandmark.LEFT_SHOULDER.value]
        right_shoulder = landmarks[mp_holistic.PoseLandmark.RIGHT_SHOULDER.value]
        frame_keypoints["neck_X"] = (left_shoulder.x + right_shoulder.x) / 2
        frame_keypoints["neck_Y"] = (left_shoulder.y + right_shoulder.y) / 2
    else:
        for name in BODY_IDENTIFIERS:
            frame_keypoints[f"{name}_X"] = 0
            frame_keypoints[f"{name}_Y"] = 0

    # --- Trích xuất keypoint BÀN TAY (Trái và Phải) ---
    for hand_side in ["left", "right"]:
        hand_landmarks = getattr(results, f"{hand_side}_hand_landmarks")
        if hand_landmarks:
            landmarks = hand_landmarks.landmark
            for name, idx in HAND_MAPPING.items():
                frame_keypoints[f"{name}_{hand_side}_X"] = landmarks[idx].x
                frame_keypoints[f"{name}_{hand_side}_Y"] = landmarks[idx].y
        else:
            for name in HAND_IDENTIFIERS:
                frame_keypoints[f"{name}_{hand_side}_X"] = 0
                frame_keypoints[f"{name}_{hand_side}_Y"] = 0
                
    return frame_keypoints


In [ ]:
def process_video_to_dataframe_row(video_path, label):
    """
    Xử lý một video, trích xuất keypoints và định dạng nó thành một dòng dữ liệu cho DataFrame.
    Bao gồm logic mới để xử lý số lượng khung hình.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Lỗi: Không thể mở video {video_path}")
        return None

    all_extracted_frames = []
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            keypoints = extract_specific_keypoints(results)
            all_extracted_frames.append(keypoints)
    cap.release()

    # --- LOGIC MỚI: ĐIỀU CHỈNH SỐ LƯỢNG KHUNG HÌNH ---
    
    # 1. Lọc bỏ các khung hình lỗi (toàn số 0)
    valid_frames = [frame for frame in all_extracted_frames if not is_zero_frame(frame)]

    final_sequence = []
    if not valid_frames:
        # Nếu video hoàn toàn không có khung hình hợp lệ, tạo chuỗi toàn 0
        print(f"\nCảnh báo: Video {video_path} không có khung hình hợp lệ nào. Tạo chuỗi 0.")
        zero_keypoints = extract_specific_keypoints(mp_holistic.Holistic().process(np.zeros((100,100,3), dtype=np.uint8))))
        final_sequence = [zero_keypoints] * TARGET_FRAMES
    
    elif len(valid_frames) >= TARGET_FRAMES:
        # 2a. Nếu có đủ hoặc thừa khung hình hợp lệ -> Lấy mẫu đều
        frame_indices = np.linspace(0, len(valid_frames) - 1, TARGET_FRAMES, dtype=int)
        final_sequence = [valid_frames[i] for i in frame_indices]
        
    else: # len(valid_frames) < TARGET_FRAMES
        # 2b. Nếu thiếu khung hình hợp lệ -> Lấy hết và lặp lại khung hình cuối
        final_sequence = valid_frames
        last_frame = valid_frames[-1]
        num_to_pad = TARGET_FRAMES - len(valid_frames)
        final_sequence.extend([last_frame] * num_to_pad)

    # --- ĐỊNH DẠNG ĐẦU RA CHO CSV ---
    row_data = {"label": label}
    for frame_num, frame_kps in enumerate(final_sequence):
        for key, value in frame_kps.items():
            column_name = f"{key}_frame{frame_num}"
            row_data[column_name] = value
            
    return row_data



In [ ]:
# lấy các video
with open('datasets/wlasl-complete/nslt_100.json', 'r') as f:
    asl100 = json.load(f)

In [ ]:
# --- HÀM CHÍNH ĐỂ CHẠY ---
def main():
    videos_to_process = [
        ("path/to/your/video1.mp4", "ky_hieu_so_1"),
        ("path/to/your/video2.mp4", "ky_hieu_so_2"),
        # Thêm các video khác ở đây...
    ]
    output_csv_path = "extracted_keypoints_dataset_v2.csv"
    all_video_rows = []
    print("Bắt đầu quá trình trích xuất keypoint với logic mới...")
    
    for video_path, label in tqdm(videos_to_process, desc="Đang xử lý các video"):
        if not os.path.exists(video_path):
            print(f"\nCảnh báo: Không tìm thấy video tại '{video_path}'. Bỏ qua.")
            continue
        row = process_video_to_dataframe_row(video_path, label)
        if row:
            all_video_rows.append(row)
    
    if not all_video_rows:
        print("Không có video nào được xử lý thành công. Kết thúc.")
        return

    df = pd.DataFrame(all_video_rows)
    cols = df.columns.tolist()
    if 'label' in cols:
        cols.remove('label')
        cols.sort()
        final_cols = ['label'] + cols
        df = df[final_cols]

    df.to_csv(output_csv_path, index=False)
    
    print(f"\nQuá trình hoàn tất! Dữ liệu đã được lưu tại: {output_csv_path}")
    print(f"Tổng số video đã xử lý: {len(df)}")
    print(f"Tổng số cột trong file CSV: {len(df.columns)}")

if __name__ == "__main__":
    main()

In [1]:
import pandas as pd
import numpy as np
import ast  # Dùng để chuyển đổi chuỗi thành danh sách một cách an toàn
from tqdm import tqdm

# --- CẤU HÌNH ---
INPUT_CSV_PATH = "datasets/inputs/VisionAPI_WLASL100/WLASL100_test_25fps.csv"  # <-- THAY ĐỔI ĐƯỜNG DẪN NÀY
OUTPUT_CSV_PATH = "processed_nested_dataset_204frames.csv" # Tên tệp đầu ra
TARGET_FRAMES = 204
# ------------------

def safe_literal_eval(val):
    """
    Hàm chuyển đổi chuỗi thành danh sách Python một cách an toàn.
    Xử lý các trường hợp chuỗi không hợp lệ hoặc rỗng.
    """
    if isinstance(val, str) and val.startswith('[') and val.endswith(']'):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            # Nếu chuỗi không phải là một danh sách hợp lệ, trả về list rỗng
            return []
    # Nếu giá trị đã là list hoặc không phải dạng chuỗi mong muốn
    return val if isinstance(val, list) else []

def row_to_frames_list(row):
    """
    Chuyển đổi một hàng của DataFrame (định dạng lồng nhau) thành một danh sách các khung hình.
    """
    # Lấy tên các cột keypoint
    keypoint_cols = [col for col in row.index if col != 'labels'] # Giả sử có cột 'labels'
    
    # Chuyển đổi tất cả các ô sang dạng list
    parsed_data = {col: safe_literal_eval(row[col]) for col in keypoint_cols}
    
    # Tìm độ dài chuỗi dài nhất để xác định số khung hình
    try:
        num_frames = max(len(v) for v in parsed_data.values() if isinstance(v, list))
    except ValueError:
        return [] # Nếu không có dữ liệu nào

    frames = []
    for i in range(num_frames):
        frame_data = {}
        for col in keypoint_cols:
            # Lấy giá trị tại khung hình i, nếu không có thì dùng NaN
            frame_data[col] = parsed_data[col][i] if i < len(parsed_data[col]) else np.nan
        frames.append(frame_data)
        
    return frames

def is_invalid_frame(frame_dict):
    """
    Kiểm tra xem một khung hình có phải là không hợp lệ hay không
    (tất cả giá trị là 0 hoặc NaN).
    """
    for value in frame_dict.values():
        if pd.notna(value) and value != 0:
            return False
    return True

def resample_frames(frames_list):
    """
    Áp dụng logic lấy mẫu lại để có được đúng TARGET_FRAMES.
    """
    valid_frames = [frame for frame in frames_list if not is_invalid_frame(frame)]
    
    final_sequence = []
    
    if not valid_frames:
        if frames_list:
            zero_frame = {key: 0 for key in frames_list[0].keys()}
        else:
            return []
        final_sequence = [zero_frame] * TARGET_FRAMES
    elif len(valid_frames) >= TARGET_FRAMES:
        indices_to_sample = np.linspace(0, len(valid_frames) - 1, TARGET_FRAMES, dtype=int)
        final_sequence = [valid_frames[i] for i in indices_to_sample]
    else:
        final_sequence = valid_frames
        last_valid_frame = valid_frames[-1]
        num_to_pad = TARGET_FRAMES - len(valid_frames)
        final_sequence.extend([last_valid_frame] * num_to_pad)
        
    return final_sequence

def frames_list_to_row(frames_list, original_columns):
    """
    Chuyển đổi danh sách khung hình đã xử lý trở lại định dạng hàng lồng nhau.
    """
    if not frames_list:
        return {col: [] for col in original_columns}

    # Khởi tạo một dictionary với các list rỗng
    new_row = {key: [] for key in frames_list[0].keys()}
    
    for frame in frames_list:
        for key, value in frame.items():
            new_row[key].append(value)
            
    return new_row

def main():
    print(f"Đang đọc tệp CSV từ: {INPUT_CSV_PATH}")
    try:
        df = pd.read_csv(INPUT_CSV_PATH)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy tệp tại '{INPUT_CSV_PATH}'. Vui lòng kiểm tra lại đường dẫn.")
        return

    processed_rows = []
    
    print(f"Bắt đầu xử lý {len(df)} video để chuẩn hóa về {TARGET_FRAMES} khung hình...")
    
    # Xác định các cột keypoint để xử lý
    keypoint_columns = [col for col in df.columns if col not in ['labels', 'video_fps', 'video_size_width', 'video_size_height']] # Loại trừ các cột meta

    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Đang xử lý"):
        # Bước 1: Tái cấu trúc hàng thành danh sách các khung hình
        frames = row_to_frames_list(row[keypoint_columns])
        
        # Bước 2: Áp dụng logic lấy mẫu lại
        resampled_sequence = resample_frames(frames)
        
        # Bước 3: Chuyển đổi chuỗi đã xử lý trở lại định dạng hàng lồng nhau
        new_row_data = frames_list_to_row(resampled_sequence, keypoint_columns)
        
        # Thêm lại các cột meta không được xử lý
        for col in df.columns:
            if col not in keypoint_columns:
                new_row_data[col] = row[col]

        processed_rows.append(new_row_data)

    # Tạo DataFrame mới
    new_df = pd.DataFrame(processed_rows)
    # Đảm bảo thứ tự cột giống file gốc
    new_df = new_df[df.columns]
    
    # Lưu kết quả
    new_df.to_csv(OUTPUT_CSV_PATH, index=False)
    
    print("\n--- HOÀN TẤT ---")
    print(f"Đã xử lý thành công và lưu tệp mới tại: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    main()

Đang xử lý:   0%|          | 0/258 [00:00<?, ?it/s]

Đang đọc tệp CSV từ: datasets/inputs/VisionAPI_WLASL100/WLASL100_test_25fps.csv
Bắt đầu xử lý 258 video để chuẩn hóa về 204 khung hình...


Đang xử lý: 100%|██████████| 258/258 [00:05<00:00, 46.63it/s]



--- HOÀN TẤT ---
Đã xử lý thành công và lưu tệp mới tại: processed_nested_dataset_204frames.csv


In [2]:
df_204=pd.read_csv(OUTPUT_CSV_PATH)

In [3]:
df_204

,indexMCP_right_X,middleMCP_left_X,ringPIP_right_Y,thumbIP_left_X,ringDIP_left_Y,littleMCP_right_Y,indexDIP_right_Y,thumbCMC_left_Y,thumbCMC_right_Y,middleTip_right_Y,...,thumbIP_right_Y,neck_X,indexMCP_left_X,indexPIP_left_X,littleMCP_left_Y,thumbIP_right_X,indexPIP_right_Y,thumbIP_left_Y,labels,indexTip_left_Y
0,"[0.398577, 0.398577, 0.398577, 0, 0, 0, 0.3985...","[0.492837, 0.492837, 0.492837, 0, 0, 0, 0.4928...","[0.528982, 0.528982, 0.528982, 0, 0, 0, 0.5289...","[0.471492, 0.471492, 0.471492, 0, 0, 0, 0.4714...","[0.0485368, 0.0485368, 0.0485368, 0, 0, 0, 0.0...","[0.444718, 0.444718, 0.444718, 0, 0, 0, 0.4447...","[0.595102, 0.595102, 0.595102, 0, 0, 0, 0.5951...","[0.171264, 0.171264, 0.171264, 0, 0, 0, 0.1712...","[0.473231, 0.473231, 0.473231, 0, 0, 0, 0.4732...","[0.536177, 0.536177, 0.536177, 0, 0, 0, 0.5361...",...,"[0.587632, 0.587632, 0.587632, 0, 0, 0, 0.5876...","[0.480291, 0.480291, 0.480291, 0.476794, 0.476...","[0.487587, 0.487587, 0.487587, 0, 0, 0, 0.4875...","[0.441586, 0.441586, 0.441586, 0, 0, 0, 0.4415...","[0.0318525, 0.0318525, 0.0318525, 0, 0, 0, 0.0...","[0.388652, 0.388652, 0.388652, 0, 0, 0, 0.3886...","[0.58603, 0.58603, 0.58603, 0, 0, 0, 0.58603, ...","[0.227427, 0.227427, 0.227427, 0, 0, 0, 0.2274...",61,"[0.145215, 0.145215, 0.145215, 0, 0, 0, 0.1452..."
1,"[0.408696, 0.415961, 0.415961, 0.408696, 0.415...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.468698, 0.656023, 0.656023, 0.468698, 0.656...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.399201, 0.598355, 0.598355, 0.399201, 0.598...","[0.530264, 0.704665, 0.704665, 0.530264, 0.704...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.432256, 0.604567, 0.604567, 0.432256, 0.604...","[0.545775, 0.705192, 0.705192, 0.545775, 0.705...",...,"[0.494947, 0.675244, 0.675244, 0.494947, 0.675...","[0.490255, 0.485767, 0.485767, 0.490255, 0.485...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.421511, 0.429935, 0.429935, 0.421511, 0.429...","[0.497688, 0.697756, 0.697756, 0.497688, 0.697...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",61,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5...","[0.503545, 0.503743, 0.503743, 0.503743, 0.503...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.4...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",95,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"[0.515727, 0.515727, 0.497205, 0.493492, 0.505...","[0.491399, 0.491399, 0.52965, 0.537391, 0.5394...","[0.198921, 0.198921, 0.120439, 0.189045, 0.195...","[0.479363, 0.479363, 0.537088, 0.49038, 0.4865...","[0.12111, 0.12111, 0.186286, 0.123821, 0.12318...","[0.204137, 0.204137, 0.139304, 0.216653, 0.216...","[0.237712, 0.237712, 0.165377, 0.229933, 0.233...","[0.194446, 0.194446, 0.202349, 0.184464, 0.183...","[0.199593, 0.199593, 0.193439, 0.229344, 0.221...","[0.205773, 0.205773, 0.133002, 0.206477, 0.188...",...,"[0.23308, 0.23308, 0.18231, 0.241729, 0.24019,...","[0.513808, 0.513808, 0.513819, 0.513894, 0.513...","[0.493364, 0.493364, 0.52099, 0.536948, 0.5388...","[0.493202, 0